In [1]:
# This Python code perfectly reproduces the caret::train function in R with the following specifications
# -R code------
# train_control <- caret::trainControl(method = 'cv', number = 10, savePredictions = 'all', seeds = 2024)
# tune_grid <- expand.grid(
#   mtry = 3,                     
#   splitrule = 'extratrees',
#   min.node.size = 50
# )

# rf_full_model <- caret::train(
#   farm_area_ha ~ .,
#   data = lsms_spatial,
#   method = 'ranger',
#   trainControl = train_control,
#   keep.inbag = T,
#   tuneGrid = tune_grid,
#   importance  = 'permutation', # how to get this in Python??
#   metric = 'RMSE',
#   min.bucket = 20,
#   num.trees = 1500
# )

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

import rasterio
import joblib
import time
# ---------------------------------------------------------------------------------------------
# Load table and light data wrangling
lsms_spatial = pd.read_csv('../data/processed/lsms_spatial_with_country_names.csv')     # this is with the avg of 3 cropland rasters


# lsms_spatial = lsms_spatial[~ lsms_spatial['country'] .isin (['Ghana', 'Rwanda'])]
lsms_spatial = lsms_spatial[['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market']]
lsms_spatial = lsms_spatial.dropna()
# lsms_spatial = lsms_spatial[:1000] # uncomment to code faster
print(lsms_spatial.columns)
print(lsms_spatial)

# define input and output
X = lsms_spatial.drop(columns = ['farm_area_ha'])
y = lsms_spatial['farm_area_ha']
# -----------------------------------------------------------------------------------------------




# Random forest models with and without extra-trees regressor
print('---------------------Without extra-tree regressor------------------------------')
deb = time.time()
# train RF without splitrule = 'extratrees'
rf = RandomForestRegressor(
    n_estimators = 1500, 
    criterion = 'squared_error', 
    min_samples_split = 50,
    min_samples_leaf = 20, 
    max_features = 3, 
    oob_score = True, 
    bootstrap = True, 
    random_state = 2024
)

# Perform cross-validation to get CV R-squared
cv_scores_rf = cross_val_score(rf, X, y, cv = 10, scoring = 'r2', n_jobs = -1)
cv_r2_rf = cv_scores_rf.mean()

# Fit the model to get OOB R-squared
rf.fit(X, y)
oob_r2_rf = rf.oob_score_
fin = time.time()
print(f"RF training time:  {fin - deb} seconds")

print("Simple RF CV R-squared: ", cv_r2_rf)
print("Simple RF OOB R-squared: ", oob_r2_rf)



print('---------------------Extra-tree regressor------------------------------')
start_time = time.time()
# Using extra-trees arguments in the RF
# Define the parameter grid
param_grid = {
    'max_features': [3],       # equivalent to mtry = 3
    'min_samples_split': [50], # equivalent to min.node.size = 50
    'min_samples_leaf': [20],  # equivalent to min.bucket = 20
    'n_estimators': [1500]     # number of trees
}

# Initialize the ExtraTreesRegressor
# etr = ExtraTreesRegressor(criterion = 'squared_error', random_state = 2024)
etr = ExtraTreesRegressor(
    criterion = 'squared_error', 
    oob_score=True, 
    bootstrap=True, 
    random_state = 2024
)

# Step 1: Perform over-all cross-validation (to evaluate the approach, not just for the best hyper-parameterized model)
cv_scores_etr = cross_val_score(etr, X, y, cv = 10, scoring = 'r2', n_jobs = -1)
cv_r2_etr = cv_scores_etr.mean()
print("Overall ExtraTreesRegressor CV R-squared: ", {cv_r2_etr})

# Step 2: Model selection
# Perform grid search
grid_search = GridSearchCV(estimator = etr, param_grid = param_grid, cv = 10, n_jobs = -1, verbose = 2)

# Fit model
grid_search.fit(X, y)
end_time = time.time()
print(f"Extra-trees training time: {end_time - start_time} seconds")

# Get the best model
best_model = grid_search.best_estimator_
print("Best Extra-trese Model:", best_model)

# Get the best hyperparameters
best_params = grid_search.best_params_
print("Best Extra-trees Hyperparameters:", best_params)

# Get the best score
best_score = grid_search.best_score_
print("Best Extra-trees Score:", best_score)

# Get the OOB R square for the best model
oob_rsquare = best_model.oob_score_
print("Extra-trees OOB R2 of best Model:", oob_rsquare)

# Calculate the cross-validation (CV) R-squared value
cv_r2 = cross_val_score(best_model, X, y, cv = 10, scoring = 'r2').mean()
print(f"Extra-trees CV R2 of the best model: {cv_r2}")


Index(['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market'],
      dtype='object')
        farm_area_ha     cropland       cattle        pop  \
0           0.095855     8.672513   785.641602  81.996536   
1           2.000000     8.672513   785.641602  81.996536   
2           0.223116     8.672513   785.641602  81.996536   
3           3.064620     8.672513   785.641602  81.996536   
4           0.141745     8.672513   785.641602  81.996536   
...              ...          ...          ...        ...   
166550      0.493900   423.718079  1723.708008  63.164284   
166551      0.263200   423.718079  1723.708008  63.164284   
166552      0.170000   423.718079  1723.708008  63.164284   
166553      0.404858  2877.621338  7079.853516  97.395699   
166558      0.001500     3.996233   407.219055   3.304352   

        cropland_per_capita       sand     slope  temperature     rainfall  \
0               

In [2]:
# --------------------------------------------------------------------------------
# predict raster
# Load the input raster
input_file = '../data/processed/stacked_rasters_africa.tif'
with rasterio.open(input_file) as src:
    input_raster = src.read()  # Read all bands
    profile = src.profile

# Reshape the raster data for prediction
n_bands, height, width = input_raster.shape
input_raster_reshaped = input_raster.reshape(n_bands, -1).T  # Reshape to (n_samples, n_features)

# Filter out rows with NaN values
valid_mask = ~np.isnan(input_raster_reshaped).any(axis=1)
input_raster_valid = input_raster_reshaped[valid_mask]

# Predict using the loaded RF model
rf_output_valid = rf.predict(input_raster_valid)  # RF without extra-trees

# Create an output array and fill with NaNs
rf_output_raster = np.full((height * width,), np.nan)
rf_output_raster[valid_mask] = rf_output_valid
rf_output_raster = rf_output_raster.reshape(height, width)

# Update the profile for the RF output raster
rf_profile = profile.copy()
rf_profile.update(count=1)

# Write the RF output raster
output_rf_file = '../data/processed/Pythom_3rast_rf_predictions_africa.tif'
with rasterio.open(output_rf_file, 'w', **rf_profile) as dst:
    dst.write(rf_output_raster, 1)

C:\Users\DHOUGNI\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [3]:
# --------------------------------------------------------------------------------
# predict raster
# Load the input raster
input_file = '../data/processed/stacked_rasters_africa.tif'
with rasterio.open(input_file) as src:
    input_raster = src.read()  # Read all bands
    profile = src.profile

# Reshape the raster data for prediction
n_bands, height, width = input_raster.shape
input_raster_reshaped = input_raster.reshape(n_bands, -1).T  # Reshape to (n_samples, n_features)

# Filter out rows with NaN values
valid_mask = ~np.isnan(input_raster_reshaped).any(axis=1)
input_raster_valid = input_raster_reshaped[valid_mask]

# Predict using the loaded RF model
rf_output_valid = best_model.predict(input_raster_valid)  # RF with extra-trees

# Create an output array and fill with NaNs
rf_output_raster = np.full((height * width,), np.nan)
rf_output_raster[valid_mask] = rf_output_valid
rf_output_raster = rf_output_raster.reshape(height, width)

# Update the profile for the RF output raster
rf_profile = profile.copy()
rf_profile.update(count=1)

# Write the RF output raster
output_rf_file = '../data/processed/Python_3rast_rf_predictions_africa.tif'
with rasterio.open(output_rf_file, 'w', **rf_profile) as dst:
    dst.write(rf_output_raster, 1)

C:\Users\DHOUGNI\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but ExtraTreesRegressor was fitted with feature names
  warnings.warn(


In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

import rasterio
import joblib
import time
# ---------------------------------------------------------------------------------------------
# Load table and light data wrangling
lsms_spatial = pd.read_csv('../data/processed/lsms_spatial_potapov2019.csv')          # this is with GLAD 2019 (Potatpov) as cropland


# lsms_spatial = lsms_spatial[~ lsms_spatial['country'] .isin (['Ghana', 'Rwanda'])]
lsms_spatial = lsms_spatial[['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market']]
lsms_spatial = lsms_spatial.dropna()
print(lsms_spatial.columns)
print(lsms_spatial)

# define input and output
X = lsms_spatial.drop(columns = ['farm_area_ha'])
y = lsms_spatial['farm_area_ha']
# -----------------------------------------------------------------------------------------------




# Random forest models with and without extra-trees regressor
print('---------------------Without extra-tree regressor------------------------------')
deb = time.time()
# train RF without splitrule = 'extratrees'
rf = RandomForestRegressor(
    n_estimators = 1500, 
    criterion = 'squared_error', 
    min_samples_split = 50,
    min_samples_leaf = 20, 
    max_features = 3, 
    oob_score = True, 
    bootstrap = True, 
    random_state = 2024
)

# Perform cross-validation to get CV R-squared
cv_scores_rf = cross_val_score(rf, X, y, cv = 10, scoring = 'r2', n_jobs = -1)
cv_r2_rf = cv_scores_rf.mean()

# Fit the model to get OOB R-squared
rf.fit(X, y)
oob_r2_rf = rf.oob_score_
fin = time.time()
print(f"RF training time:  {fin - deb} seconds")

print("Simple RF CV R-squared: ", cv_r2_rf)
print("Simple RF OOB R-squared: ", oob_r2_rf)



print('---------------------Extra-tree regressor------------------------------')
start_time = time.time()
# Using extra-trees arguments in the RF
# Define the parameter grid
param_grid = {
    'max_features': [3],       # equivalent to mtry = 3
    'min_samples_split': [50], # equivalent to min.node.size = 50
    'min_samples_leaf': [20],  # equivalent to min.bucket = 20
    'n_estimators': [1500]     # number of trees
}

# Initialize the ExtraTreesRegressor
etr = ExtraTreesRegressor(
    criterion = 'squared_error', 
    oob_score=True, 
    bootstrap=True, 
    random_state = 2024
)

# Step 1: Perform over-all cross-validation (to evaluate the approach, not just for the best hyper-parameterized model)
cv_scores_etr = cross_val_score(etr, X, y, cv = 10, scoring = 'r2', n_jobs = -1)
cv_r2_etr = cv_scores_etr.mean()
print("Overall ExtraTreesRegressor CV R-squared: ", {cv_r2_etr})

# Step 2: Model selection
# Perform grid search
grid_search = GridSearchCV(estimator = etr, param_grid = param_grid, cv = 10, n_jobs = -1, verbose = 2)

# Fit model
grid_search.fit(X, y)
end_time = time.time()
print(f"Extra-trees training time: {end_time - start_time} seconds")

# Get the best model
best_model = grid_search.best_estimator_
print("Best Extra-trese Model:", best_model)

# Get the best hyperparameters
best_params = grid_search.best_params_
print("Best Extra-trees Hyperparameters:", best_params)

# Get the best score
best_score = grid_search.best_score_
print("Best Extra-trees Score:", best_score)

# Get the OOB R square for the best model
oob_rsquare = best_model.oob_score_
print("Extra-trees OOB R2 of best Model:", oob_rsquare)

# Calculate the cross-validation (CV) R-squared value
cv_r2 = cross_val_score(best_model, X, y, cv = 10, scoring = 'r2').mean()
print(f"Extra-trees CV R2 of the best model: {cv_r2}")



# --------------------------------------------------------------------------------
# predict raster
# Load the input raster
input_file = '../data/processed/stacked_rasters_africa.tif'
with rasterio.open(input_file) as src:
    input_raster = src.read()  # Read all bands
    profile = src.profile

# Reshape the raster data for prediction
n_bands, height, width = input_raster.shape
input_raster_reshaped = input_raster.reshape(n_bands, -1).T  # Reshape to (n_samples, n_features)

# Filter out rows with NaN values
valid_mask = ~np.isnan(input_raster_reshaped).any(axis=1)
input_raster_valid = input_raster_reshaped[valid_mask]

# Predict using the loaded RF model
rf_output_valid = best_model.predict(input_raster_valid)  # RF with extra-trees

# Create an output array and fill with NaNs
rf_output_raster = np.full((height * width,), np.nan)
rf_output_raster[valid_mask] = rf_output_valid
rf_output_raster = rf_output_raster.reshape(height, width)

# Update the profile for the RF output raster
rf_profile = profile.copy()
rf_profile.update(count=1)

# Write the RF output raster
output_rf_file = '../data/processed/Python_potapov_rf_predictions_africa.tif'
with rasterio.open(output_rf_file, 'w', **rf_profile) as dst:
    dst.write(rf_output_raster, 1)

Index(['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market'],
      dtype='object')
        farm_area_ha  cropland       cattle        pop  cropland_per_capita  \
0           0.095855  1.880119   785.641602  81.996536             0.022929   
1           2.000000  1.880119   785.641602  81.996536             0.022929   
2           0.223116  1.880119   785.641602  81.996536             0.022929   
3           3.064620  1.880119   785.641602  81.996536             0.022929   
4           0.141745  1.880119   785.641602  81.996536             0.022929   
...              ...       ...          ...        ...                  ...   
166550      0.493900  0.000000  1723.708008  63.164284             0.000000   
166551      0.263200  0.000000  1723.708008  63.164284             0.000000   
166552      0.170000  0.000000  1723.708008  63.164284             0.000000   
166553      0.404858  0.000000  7079.

C:\Users\DHOUGNI\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but ExtraTreesRegressor was fitted with feature names
  warnings.warn(


In [6]:
# This Python code perfectly reproduces the caret::train function in R with the following specifications
# -R code------
# train_control <- caret::trainControl(method = 'cv', number = 10, savePredictions = 'all', seeds = 2024)
# tune_grid <- expand.grid(
#   mtry = 3,                     
#   splitrule = 'extratrees',
#   min.node.size = 50
# )

# rf_full_model <- caret::train(
#   farm_area_ha ~ .,
#   data = lsms_spatial,
#   method = 'ranger',
#   trainControl = train_control,
#   keep.inbag = T,
#   tuneGrid = tune_grid,
#   importance  = 'permutation', # how to get this in Python??
#   metric = 'RMSE',
#   min.bucket = 20,
#   num.trees = 1500
# )

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

import rasterio
import joblib
import time
# ---------------------------------------------------------------------------------------------
# Load table and light data wrangling
lsms_spatial = pd.read_csv('../data/processed/lsms_spatial_spam2010.csv')             # this is with SPAM 2010 as cropland

lsms_spatial = lsms_spatial[['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market']]
lsms_spatial = lsms_spatial.dropna()
print(lsms_spatial.columns)
print(lsms_spatial)

# define input and output
X = lsms_spatial.drop(columns = ['farm_area_ha'])
y = lsms_spatial['farm_area_ha']
# -----------------------------------------------------------------------------------------------




# Random forest models with and without extra-trees regressor
print('---------------------Without extra-tree regressor------------------------------')
deb = time.time()
# train RF without splitrule = 'extratrees'
rf = RandomForestRegressor(
    n_estimators = 1500, 
    criterion = 'squared_error', 
    min_samples_split = 50,
    min_samples_leaf = 20, 
    max_features = 3, 
    oob_score = True, 
    bootstrap = True, 
    random_state = 2024
)

# Perform cross-validation to get CV R-squared
cv_scores_rf = cross_val_score(rf, X, y, cv = 10, scoring = 'r2', n_jobs = -1)
cv_r2_rf = cv_scores_rf.mean()

# Fit the model to get OOB R-squared
rf.fit(X, y)
oob_r2_rf = rf.oob_score_
fin = time.time()
print(f"RF training time:  {fin - deb} seconds")

print("Simple RF CV R-squared: ", cv_r2_rf)
print("Simple RF OOB R-squared: ", oob_r2_rf)



print('---------------------Extra-tree regressor------------------------------')
start_time = time.time()
# Using extra-trees arguments in the RF
# Define the parameter grid
param_grid = {
    'max_features': [3],       # equivalent to mtry = 3
    'min_samples_split': [50], # equivalent to min.node.size = 50
    'min_samples_leaf': [20],  # equivalent to min.bucket = 20
    'n_estimators': [1500]     # number of trees
}

# Initialize the ExtraTreesRegressor
etr = ExtraTreesRegressor(
    criterion = 'squared_error', 
    oob_score=True, 
    bootstrap=True, 
    random_state = 2024
)

# Step 1: Perform over-all cross-validation (to evaluate the approach, not just for the best hyper-parameterized model)
cv_scores_etr = cross_val_score(etr, X, y, cv = 10, scoring = 'r2', n_jobs = -1)
cv_r2_etr = cv_scores_etr.mean()
print("Overall ExtraTreesRegressor CV R-squared: ", {cv_r2_etr})

# Step 2: Model selection
# Perform grid search
grid_search = GridSearchCV(estimator = etr, param_grid = param_grid, cv = 10, n_jobs = -1, verbose = 2)

# Fit model
grid_search.fit(X, y)
end_time = time.time()
print(f"Extra-trees training time: {end_time - start_time} seconds")

# Get the best model
best_model = grid_search.best_estimator_
print("Best Extra-trese Model:", best_model)

# Get the best hyperparameters
best_params = grid_search.best_params_
print("Best Extra-trees Hyperparameters:", best_params)

# Get the best score
best_score = grid_search.best_score_
print("Best Extra-trees Score:", best_score)

# Get the OOB R square for the best model
oob_rsquare = best_model.oob_score_
print("Extra-trees OOB R2 of best Model:", oob_rsquare)

# Calculate the cross-validation (CV) R-squared value
cv_r2 = cross_val_score(best_model, X, y, cv = 10, scoring = 'r2').mean()
print(f"Extra-trees CV R2 of the best model: {cv_r2}")



# --------------------------------------------------------------------------------
# predict raster
# Load the input raster
input_file = '../data/processed/stacked_rasters_africa.tif'
with rasterio.open(input_file) as src:
    input_raster = src.read()  # Read all bands
    profile = src.profile

# Reshape the raster data for prediction
n_bands, height, width = input_raster.shape
input_raster_reshaped = input_raster.reshape(n_bands, -1).T  # Reshape to (n_samples, n_features)

# Filter out rows with NaN values
valid_mask = ~np.isnan(input_raster_reshaped).any(axis=1)
input_raster_valid = input_raster_reshaped[valid_mask]

# Predict using the loaded RF model
rf_output_valid = best_model.predict(input_raster_valid)  # RF with extra-trees

# Create an output array and fill with NaNs
rf_output_raster = np.full((height * width,), np.nan)
rf_output_raster[valid_mask] = rf_output_valid
rf_output_raster = rf_output_raster.reshape(height, width)

# Update the profile for the RF output raster
rf_profile = profile.copy()
rf_profile.update(count=1)

# Write the RF output raster
output_rf_file = '../data/processed/Python_SPAM2010_rf_predictions_africa.tif'
with rasterio.open(output_rf_file, 'w', **rf_profile) as dst:
    dst.write(rf_output_raster, 1)

Index(['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market'],
      dtype='object')
        farm_area_ha     cropland       cattle        pop  \
0           0.095855  1631.099976   785.641602  81.996536   
1           2.000000  1631.099976   785.641602  81.996536   
2           0.223116  1631.099976   785.641602  81.996536   
3           3.064620  1631.099976   785.641602  81.996536   
4           0.141745  1631.099976   785.641602  81.996536   
...              ...          ...          ...        ...   
166549      0.457500  3761.899902  1723.708008  63.164284   
166550      0.493900  3761.899902  1723.708008  63.164284   
166551      0.263200  3761.899902  1723.708008  63.164284   
166552      0.170000  3761.899902  1723.708008  63.164284   
166553      0.404858  2044.799927  7079.853516  97.395699   

        cropland_per_capita       sand     slope  temperature     rainfall  \
0               

C:\Users\DHOUGNI\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but ExtraTreesRegressor was fitted with feature names
  warnings.warn(


In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

import rasterio
import joblib
import time
# ---------------------------------------------------------------------------------------------
# Load table and light data wrangling
lsms_spatial = pd.read_csv('../data/processed/lsms_spatial_spam2017.csv')             # this is with SPAM 2017 as cropland

# lsms_spatial = lsms_spatial[~ lsms_spatial['country'] .isin (['Ghana', 'Rwanda'])]
lsms_spatial = lsms_spatial[['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market']]
lsms_spatial = lsms_spatial.dropna()
print(lsms_spatial.columns)
print(lsms_spatial)

# define input and output
X = lsms_spatial.drop(columns = ['farm_area_ha'])
y = lsms_spatial['farm_area_ha']
# -----------------------------------------------------------------------------------------------




# Random forest models with and without extra-trees regressor
print('---------------------Without extra-tree regressor------------------------------')
deb = time.time()
# train RF without splitrule = 'extratrees'
rf = RandomForestRegressor(
    n_estimators = 1500, 
    criterion = 'squared_error', 
    min_samples_split = 50,
    min_samples_leaf = 20, 
    max_features = 3, 
    oob_score = True, 
    bootstrap = True, 
    random_state = 2024
)

# Perform cross-validation to get CV R-squared
cv_scores_rf = cross_val_score(rf, X, y, cv = 10, scoring = 'r2', n_jobs = -1)
cv_r2_rf = cv_scores_rf.mean()

# Fit the model to get OOB R-squared
rf.fit(X, y)
oob_r2_rf = rf.oob_score_
fin = time.time()
print(f"RF training time:  {fin - deb} seconds")

print("Simple RF CV R-squared: ", cv_r2_rf)
print("Simple RF OOB R-squared: ", oob_r2_rf)



print('---------------------Extra-tree regressor------------------------------')
start_time = time.time()
# Using extra-trees arguments in the RF
# Define the parameter grid
param_grid = {
    'max_features': [3],       # equivalent to mtry = 3
    'min_samples_split': [50], # equivalent to min.node.size = 50
    'min_samples_leaf': [20],  # equivalent to min.bucket = 20
    'n_estimators': [1500]     # number of trees
}

# Initialize the ExtraTreesRegressor
etr = ExtraTreesRegressor(
    criterion = 'squared_error', 
    oob_score=True, 
    bootstrap=True, 
    random_state = 2024
)

# Step 1: Perform over-all cross-validation (to evaluate the approach, not just for the best hyper-parameterized model)
cv_scores_etr = cross_val_score(etr, X, y, cv = 10, scoring = 'r2', n_jobs = -1)
cv_r2_etr = cv_scores_etr.mean()
print("Overall ExtraTreesRegressor CV R-squared: ", {cv_r2_etr})

# Step 2: Model selection
# Perform grid search
grid_search = GridSearchCV(estimator = etr, param_grid = param_grid, cv = 10, n_jobs = -1, verbose = 2)

# Fit model
grid_search.fit(X, y)
end_time = time.time()
print(f"Extra-trees training time: {end_time - start_time} seconds")

# Get the best model
best_model = grid_search.best_estimator_
print("Best Extra-trese Model:", best_model)

# Get the best hyperparameters
best_params = grid_search.best_params_
print("Best Extra-trees Hyperparameters:", best_params)

# Get the best score
best_score = grid_search.best_score_
print("Best Extra-trees Score:", best_score)

# Get the OOB R square for the best model
oob_rsquare = best_model.oob_score_
print("Extra-trees OOB R2 of best Model:", oob_rsquare)

# Calculate the cross-validation (CV) R-squared value
cv_r2 = cross_val_score(best_model, X, y, cv = 10, scoring = 'r2').mean()
print(f"Extra-trees CV R2 of the best model: {cv_r2}")



# --------------------------------------------------------------------------------
# predict raster
# Load the input raster
input_file = '../data/processed/stacked_rasters_africa.tif'
with rasterio.open(input_file) as src:
    input_raster = src.read()  # Read all bands
    profile = src.profile

# Reshape the raster data for prediction
n_bands, height, width = input_raster.shape
input_raster_reshaped = input_raster.reshape(n_bands, -1).T  # Reshape to (n_samples, n_features)

# Filter out rows with NaN values
valid_mask = ~np.isnan(input_raster_reshaped).any(axis=1)
input_raster_valid = input_raster_reshaped[valid_mask]

# Predict using the loaded RF model
rf_output_valid = best_model.predict(input_raster_valid)  # RF with extra-trees

# Create an output array and fill with NaNs
rf_output_raster = np.full((height * width,), np.nan)
rf_output_raster[valid_mask] = rf_output_valid
rf_output_raster = rf_output_raster.reshape(height, width)

# Update the profile for the RF output raster
rf_profile = profile.copy()
rf_profile.update(count=1)

# Write the RF output raster
output_rf_file = '../data/processed/Python_SPAM2017_rf_predictions_africa.tif'
with rasterio.open(output_rf_file, 'w', **rf_profile) as dst:
    dst.write(rf_output_raster, 1)

Index(['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market'],
      dtype='object')
        farm_area_ha     cropland       cattle        pop  \
0           0.095855  1562.800049   785.641602  81.996536   
1           2.000000  1562.800049   785.641602  81.996536   
2           0.223116  1562.800049   785.641602  81.996536   
3           3.064620  1562.800049   785.641602  81.996536   
4           0.141745  1562.800049   785.641602  81.996536   
...              ...          ...          ...        ...   
166549      0.457500  4146.000000  1723.708008  63.164284   
166550      0.493900  4146.000000  1723.708008  63.164284   
166551      0.263200  4146.000000  1723.708008  63.164284   
166552      0.170000  4146.000000  1723.708008  63.164284   
166553      0.404858  2234.000000  7079.853516  97.395699   

        cropland_per_capita       sand     slope  temperature     rainfall  \
0               

C:\Users\DHOUGNI\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but ExtraTreesRegressor was fitted with feature names
  warnings.warn(


In [8]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

import rasterio
import joblib
import time
# ---------------------------------------------------------------------------------------------
# Load table and light data wrangling
lsms_spatial = pd.read_csv('../data/processed/lsms_spatial_spam2020.csv')             # this is with SPAM 2020 as cropland

# lsms_spatial = lsms_spatial[~ lsms_spatial['country'] .isin (['Ghana', 'Rwanda'])]
lsms_spatial = lsms_spatial[['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market']]
lsms_spatial = lsms_spatial.dropna()
print(lsms_spatial.columns)
print(lsms_spatial)

# define input and output
X = lsms_spatial.drop(columns = ['farm_area_ha'])
y = lsms_spatial['farm_area_ha']
# -----------------------------------------------------------------------------------------------




# Random forest models with and without extra-trees regressor
print('---------------------Without extra-tree regressor------------------------------')
deb = time.time()
# train RF without splitrule = 'extratrees'
rf = RandomForestRegressor(
    n_estimators = 1500, 
    criterion = 'squared_error', 
    min_samples_split = 50,
    min_samples_leaf = 20, 
    max_features = 3, 
    oob_score = True, 
    bootstrap = True, 
    random_state = 2024
)

# Perform cross-validation to get CV R-squared
cv_scores_rf = cross_val_score(rf, X, y, cv = 10, scoring = 'r2', n_jobs = -1)
cv_r2_rf = cv_scores_rf.mean()

# Fit the model to get OOB R-squared
rf.fit(X, y)
oob_r2_rf = rf.oob_score_
fin = time.time()
print(f"RF training time:  {fin - deb} seconds")

print("Simple RF CV R-squared: ", cv_r2_rf)
print("Simple RF OOB R-squared: ", oob_r2_rf)



print('---------------------Extra-tree regressor------------------------------')
start_time = time.time()
# Using extra-trees arguments in the RF
# Define the parameter grid
param_grid = {
    'max_features': [3],       # equivalent to mtry = 3
    'min_samples_split': [50], # equivalent to min.node.size = 50
    'min_samples_leaf': [20],  # equivalent to min.bucket = 20
    'n_estimators': [1500]     # number of trees
}

# Initialize the ExtraTreesRegressor
etr = ExtraTreesRegressor(
    criterion = 'squared_error', 
    oob_score=True, 
    bootstrap=True, 
    random_state = 2024
)

# Step 1: Perform over-all cross-validation (to evaluate the approach, not just for the best hyper-parameterized model)
cv_scores_etr = cross_val_score(etr, X, y, cv = 10, scoring = 'r2', n_jobs = -1)
cv_r2_etr = cv_scores_etr.mean()
print("Overall ExtraTreesRegressor CV R-squared: ", {cv_r2_etr})

# Step 2: Model selection
# Perform grid search
grid_search = GridSearchCV(estimator = etr, param_grid = param_grid, cv = 10, n_jobs = -1, verbose = 2)

# Fit model
grid_search.fit(X, y)
end_time = time.time()
print(f"Extra-trees training time: {end_time - start_time} seconds")

# Get the best model
best_model = grid_search.best_estimator_
print("Best Extra-trese Model:", best_model)

# Get the best hyperparameters
best_params = grid_search.best_params_
print("Best Extra-trees Hyperparameters:", best_params)

# Get the best score
best_score = grid_search.best_score_
print("Best Extra-trees Score:", best_score)

# Get the OOB R square for the best model
oob_rsquare = best_model.oob_score_
print("Extra-trees OOB R2 of best Model:", oob_rsquare)

# Calculate the cross-validation (CV) R-squared value
cv_r2 = cross_val_score(best_model, X, y, cv = 10, scoring = 'r2').mean()
print(f"Extra-trees CV R2 of the best model: {cv_r2}")



# --------------------------------------------------------------------------------
# predict raster
# Load the input raster
input_file = '../data/processed/stacked_rasters_africa.tif'
with rasterio.open(input_file) as src:
    input_raster = src.read()  # Read all bands
    profile = src.profile

# Reshape the raster data for prediction
n_bands, height, width = input_raster.shape
input_raster_reshaped = input_raster.reshape(n_bands, -1).T  # Reshape to (n_samples, n_features)

# Filter out rows with NaN values
valid_mask = ~np.isnan(input_raster_reshaped).any(axis=1)
input_raster_valid = input_raster_reshaped[valid_mask]

# Predict using the loaded RF model
rf_output_valid = best_model.predict(input_raster_valid)  # RF with extra-trees

# Create an output array and fill with NaNs
rf_output_raster = np.full((height * width,), np.nan)
rf_output_raster[valid_mask] = rf_output_valid
rf_output_raster = rf_output_raster.reshape(height, width)

# Update the profile for the RF output raster
rf_profile = profile.copy()
rf_profile.update(count=1)

# Write the RF output raster
output_rf_file = '../data/processed/Python_SPAM2020_rf_predictions_africa.tif'
with rasterio.open(output_rf_file, 'w', **rf_profile) as dst:
    dst.write(rf_output_raster, 1)

Index(['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market'],
      dtype='object')
        farm_area_ha     cropland       cattle        pop  \
0           0.095855  1313.099976   785.641602  81.996536   
1           2.000000  1313.099976   785.641602  81.996536   
2           0.223116  1313.099976   785.641602  81.996536   
3           3.064620  1313.099976   785.641602  81.996536   
4           0.141745  1313.099976   785.641602  81.996536   
...              ...          ...          ...        ...   
166548      0.647700  5397.299805  1723.708008  63.164284   
166549      0.457500  5397.299805  1723.708008  63.164284   
166550      0.493900  5397.299805  1723.708008  63.164284   
166551      0.263200  5397.299805  1723.708008  63.164284   
166552      0.170000  5397.299805  1723.708008  63.164284   

        cropland_per_capita       sand     slope  temperature     rainfall  \
0               

C:\Users\DHOUGNI\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but ExtraTreesRegressor was fitted with feature names
  warnings.warn(


In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

import rasterio
import joblib
import time
# ---------------------------------------------------------------------------------------------
# Load table and light data wrangling
lsms_spatial = pd.read_csv('../data/processed/lsms_spatial_geosurvey2015.csv')             # this is with GEOSURVEY 2015 as cropland


# lsms_spatial = lsms_spatial[~ lsms_spatial['country'] .isin (['Ghana', 'Rwanda'])]
lsms_spatial = lsms_spatial[['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market']]
lsms_spatial = lsms_spatial.dropna()
print(lsms_spatial.columns)
print(lsms_spatial)

# define input and output
X = lsms_spatial.drop(columns = ['farm_area_ha'])
y = lsms_spatial['farm_area_ha']
# -----------------------------------------------------------------------------------------------




# Random forest models with and without extra-trees regressor
print('---------------------Without extra-tree regressor------------------------------')
deb = time.time()
# train RF without splitrule = 'extratrees'
rf = RandomForestRegressor(
    n_estimators = 1500, 
    criterion = 'squared_error', 
    min_samples_split = 50,
    min_samples_leaf = 20, 
    max_features = 3, 
    oob_score = True, 
    bootstrap = True, 
    random_state = 2024
)

# Perform cross-validation to get CV R-squared
cv_scores_rf = cross_val_score(rf, X, y, cv = 10, scoring = 'r2', n_jobs = -1)
cv_r2_rf = cv_scores_rf.mean()

# Fit the model to get OOB R-squared
rf.fit(X, y)
oob_r2_rf = rf.oob_score_
fin = time.time()
print(f"RF training time:  {fin - deb} seconds")

print("Simple RF CV R-squared: ", cv_r2_rf)
print("Simple RF OOB R-squared: ", oob_r2_rf)



print('---------------------Extra-tree regressor------------------------------')
start_time = time.time()
# Using extra-trees arguments in the RF
# Define the parameter grid
param_grid = {
    'max_features': [3],       # equivalent to mtry = 3
    'min_samples_split': [50], # equivalent to min.node.size = 50
    'min_samples_leaf': [20],  # equivalent to min.bucket = 20
    'n_estimators': [1500]     # number of trees
}

# Initialize the ExtraTreesRegressor
etr = ExtraTreesRegressor(
    criterion = 'squared_error', 
    oob_score=True, 
    bootstrap=True, 
    random_state = 2024
)

# Step 1: Perform over-all cross-validation (to evaluate the approach, not just for the best hyper-parameterized model)
cv_scores_etr = cross_val_score(etr, X, y, cv = 10, scoring = 'r2', n_jobs = -1)
cv_r2_etr = cv_scores_etr.mean()
print("Overall ExtraTreesRegressor CV R-squared: ", {cv_r2_etr})

# Step 2: Model selection
# Perform grid search
grid_search = GridSearchCV(estimator = etr, param_grid = param_grid, cv = 10, n_jobs = -1, verbose = 2)

# Fit model
grid_search.fit(X, y)
end_time = time.time()
print(f"Extra-trees training time: {end_time - start_time} seconds")

# Get the best model
best_model = grid_search.best_estimator_
print("Best Extra-trese Model:", best_model)

# Get the best hyperparameters
best_params = grid_search.best_params_
print("Best Extra-trees Hyperparameters:", best_params)

# Get the best score
best_score = grid_search.best_score_
print("Best Extra-trees Score:", best_score)

# Get the OOB R square for the best model
oob_rsquare = best_model.oob_score_
print("Extra-trees OOB R2 of best Model:", oob_rsquare)

# Calculate the cross-validation (CV) R-squared value
cv_r2 = cross_val_score(best_model, X, y, cv = 10, scoring = 'r2').mean()
print(f"Extra-trees CV R2 of the best model: {cv_r2}")



# --------------------------------------------------------------------------------
# predict raster
# Load the input raster
input_file = '../data/processed/stacked_rasters_africa.tif'
with rasterio.open(input_file) as src:
    input_raster = src.read()  # Read all bands
    profile = src.profile

# Reshape the raster data for prediction
n_bands, height, width = input_raster.shape
input_raster_reshaped = input_raster.reshape(n_bands, -1).T  # Reshape to (n_samples, n_features)

# Filter out rows with NaN values
valid_mask = ~np.isnan(input_raster_reshaped).any(axis=1)
input_raster_valid = input_raster_reshaped[valid_mask]

# Predict using the loaded RF model
rf_output_valid = best_model.predict(input_raster_valid)  # RF with extra-trees

# Create an output array and fill with NaNs
rf_output_raster = np.full((height * width,), np.nan)
rf_output_raster[valid_mask] = rf_output_valid
rf_output_raster = rf_output_raster.reshape(height, width)

# Update the profile for the RF output raster
rf_profile = profile.copy()
rf_profile.update(count=1)

# Write the RF output raster
output_rf_file = '../data/processed/Python_Geosurvey2015_rf_predictions_africa.tif'
with rasterio.open(output_rf_file, 'w', **rf_profile) as dst:
    dst.write(rf_output_raster, 1)

Index(['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market'],
      dtype='object')
        farm_area_ha   cropland       cattle        pop  cropland_per_capita  \
0           0.095855  37.344075   785.641602  81.996536             0.455435   
1           2.000000  37.344075   785.641602  81.996536             0.455435   
2           0.223116  37.344075   785.641602  81.996536             0.455435   
3           3.064620  37.344075   785.641602  81.996536             0.455435   
4           0.141745  37.344075   785.641602  81.996536             0.455435   
...              ...        ...          ...        ...                  ...   
166550      0.493900  48.283944  1723.708008  63.164284             0.764418   
166551      0.263200  48.283944  1723.708008  63.164284             0.764418   
166552      0.170000  48.283944  1723.708008  63.164284             0.764418   
166553      0.404858  47.79

C:\Users\DHOUGNI\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but ExtraTreesRegressor was fitted with feature names
  warnings.warn(
